# Get to Know a Dataset: ECMWF IFS Analysis N320 - Beyond Weather Icechunk Zarr

This notebook serves as a guided tour of the [ECMWF IFS Analysis N320 - Beyond Weather Icechunk Zarr](https://registry.opendata.aws/beyond-weather-ecmwf-ifs-analysis/) dataset, published by [Beyond Weather](https://beyond-weather.com). More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

The dataset contains the operational analysis of ECMWF's Integrated Forecasting System (IFS), retrieved from the ECMWF MARS archive every 6 hours (00, 06, 12 and 18 UTC) from 2011-01-01 to 2024-12-31. This is the analysis that ECMWF itself uses to initialise its operational high-resolution forecasts: the best estimate of the state of the atmosphere at each time, produced by 4D-Var data assimilation of observations into the IFS model. It is neither the ERA5 reanalysis (which uses a single, fixed model version and is produced separately) nor the initial conditions of the IFS ENS ensemble[^1]. The data is stored on the native N320 reduced Gaussian grid (542,080 points, roughly 31 km spacing) with 18 pressure levels, as a single cloud-native [Icechunk](https://icechunk.io) / [Zarr v3](https://zarr.dev) store. The variable set matches the inputs of data-driven weather models such as ECMWF's [AIFS](https://www.ecmwf.int/en/forecasts/dataset/aifs-machine-learning-data), so the data can be used to train, fine-tune, initialise and verify them.

[^1]: The IFS ENS step-0 (t0) fields are derived from, but not identical to, this analysis. The operational analysis is a single high-resolution 4D-Var assimilation of observations, and it is the analysis that initialises the deterministic high-resolution forecast (formerly HRES). The control member of IFS ENS starts from this analysis. The perturbed members start from the control plus perturbations taken from the Ensemble of Data Assimilations (EDA), a set of lower-resolution 4D-Var runs with perturbed observations, and from singular vectors. Since IFS Cycle 50r1 (12 May 2026), HRES no longer exists as a separate product: the single unperturbed forecast is now the ENS control. For 2011–2024, the period covered here, HRES still refers to the former deterministic forecast. This archive is also not temporally homogeneous: the operational IFS is upgraded several times a year, so changes in model version, resolution and observation usage can cause small discontinuities, unlike a reanalysis such as ERA5.

### Structure of the Dataset

At the top level of the `beyond-weather-ecmwf-ifs-analysis` bucket you will find:

1. `README.md`: a short description of the dataset with links to the full documentation
2. `LICENSE.txt`: the licence and attribution requirements
3. `data/`: one prefix per dataset. Today this holds a single dataset, `data/ifs-analysis-n320/`. Other grids or variable sets may be added later as siblings.

Each dataset prefix is an **Icechunk repository**. Icechunk is a transactional storage engine for Zarr: it stores the Zarr chunks plus manifests and snapshots that record which chunks belong to which version of the data. Inside the repository you will see the prefixes `chunks/`, `manifests/`, `snapshots/`, `transactions/` and `refs/`, plus a `repo` object. Object names under `chunks/` are content IDs, not human-readable paths. **Do not read these objects directly.** Open the repository with the `icechunk` Python library and use the Zarr/xarray view it gives you (shown below).

Full documentation, including a data dictionary, is at: https://github.com/Beyond-Weather-Git/open-data-examples/blob/main/ifs-analysis-n320/README.md

In [ ]:
import boto3
import icechunk as ic
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from botocore import UNSIGNED
from botocore.config import Config

In [ ]:
# Location of the S3 bucket for this dataset
BUCKET = "beyond-weather-ecmwf-ifs-analysis"
REGION = "us-east-2"
PREFIX = "data/ifs-analysis-n320"

# Public bucket: use unsigned requests
s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))

response = s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")
for item in response.get("Contents", []):
    print(item["Key"])
for item in response.get("CommonPrefixes", []):
    print(item["Prefix"])

Looking inside the dataset prefix shows the Icechunk repository layout described above.

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIX}/", Delimiter="/")
for item in response.get("Contents", []):
    print(item["Key"])
for item in response.get("CommonPrefixes", []):
    print(item["Prefix"])

### Data formats

The data is a **Zarr v3** store managed by **Icechunk**.

- **Zarr** is a chunked, compressed, N-dimensional array format designed for cloud object storage. Readers fetch only the chunks they need, using parallel HTTP range requests. xarray, dask and most of the Pangeo ecosystem can read it.
- **Icechunk** adds version control and ACID transactions on top of Zarr. Each update to the dataset (for example, appending new dates) is committed as an immutable *snapshot*. You can open the latest version on the `main` branch, or pin your analysis to a specific snapshot ID so it stays reproducible. Icechunk needs the `icechunk` Python package. Plain `zarr.open("s3://...")` will not work.

Details specific to this dataset:

- **Grid.** Values are stored on ECMWF's **N320 reduced Gaussian grid** as a flat `point` dimension of 542,080 points, not as a lat/lon rectangle. The `latitude` and `longitude` coordinates give each point's location. Points run north to south in rows of constant latitude (640 rows), and each row goes eastward from 0°E. The number of points per row shrinks towards the poles, so grid spacing stays roughly uniform (~31 km). To get a regular lat/lon grid, use a regridding tool such as [earthkit-regrid](https://github.com/ecmwf/earthkit-regrid). Many analyses (zonal means, area averages, ML models on graphs or meshes) can work on the native points directly.
- **Dimensions.** `time` (6-hourly, 20,456 steps), `pressure_level` (18 levels from 1000 to 2 hPa, for upper-air variables) and `point`.
- **Chunking.** One chunk holds one timestep of one variable over the whole globe (`(1, 542080)` for surface fields, `(1, 18, 542080)` for upper-air fields). Reading global fields at a few times is cheap. Reading a long time series at a single location means touching one chunk per timestep. In that case, read in parallel with dask and run your code close to the data (in `us-east-2`).
- **Compression.** Chunks are compressed with Blosc/Zstandard after **bit rounding**. Each variable keeps only the number of mantissa bits listed in its `bwdl_keepbits` attribute, chosen to stay well below the analysis uncertainty. The values are therefore *lossy* float32 approximations of the GRIB values from MARS.
- **"Static" fields.** `land_sea_mask`, `orography`, `standard_deviation_of_orography` and `slope_of_sub_gridscale_orography` are stored at every timestep, because ECMWF changes them when the IFS model is upgraded (see the question section below).
- **Metadata.** Each variable keeps the GRIB metadata from MARS as attributes (`GRIB_shortName`, `GRIB_paramId`, `units`, ...), so you can look it up in the [ECMWF parameter database](https://codes.ecmwf.int/grib/param-db/).
- **IFS cycles.** This archive is a concatenation of operational IFS analysis runs. Some variables or definitions have changed between cycles. Check: [IFS cycle change history](https://confluence.ecmwf.int/display/FCST/Changes+to+the+forecasting+system).

### How to load and download the dataset

We open the Icechunk repository anonymously, start a read-only session on the `main` branch, and hand its Zarr store to xarray. This step only reads metadata. Data is loaded lazily when you ask for it.

In [ ]:
storage = ic.s3_storage(bucket=BUCKET, prefix=PREFIX, region=REGION, anonymous=True)
repo = ic.Repository.open(storage)
session = repo.readonly_session(branch="main")

# Note the snapshot id if you want to come back to exactly this version later:
# repo.readonly_session(snapshot_id=...)
print("Snapshot:", session.snapshot_id)

ds = xr.open_zarr(session.store)
ds

Every variable carries its GRIB metadata and the number of mantissa bits kept after bit rounding. Here are a few of them as a small data dictionary:

In [ ]:
for name in ["2m_temperature", "mean_sea_level_pressure", "temperature", "specific_humidity", "orography"]:
    attrs = ds[name].attrs
    print(
        f"{name:28s} shortName={attrs['GRIB_shortName']:4s} paramId={attrs['GRIB_paramId']:<7} "
        f"units={attrs['units']:12s} level={attrs['GRIB_typeOfLevel']:20s} keepbits={attrs['bwdl_keepbits']}"
    )

### Visual inspection!

In [ ]:
t2m = ds["2m_temperature"].sel(time="2024-07-15T12:00").load()
latitude = ds["latitude"].values
# Shift longitudes from [0, 360) to [-180, 180) for a Greenwich-centred map
lon_centred = ((ds["longitude"].values + 180) % 360) - 180

fig, ax = plt.subplots(figsize=(14, 7), dpi=100)
scatter = ax.scatter(lon_centred, latitude, c=t2m.values - 273.15, s=0.3, cmap="RdYlBu_r",
                     vmin=-40, vmax=45, rasterized=True)
fig.colorbar(scatter, ax=ax, label="2 m temperature (°C)", shrink=0.8)
ax.set(xlim=(-180, 180), ylim=(-90, 90), xlabel="longitude", ylabel="latitude",
       title="IFS operational analysis, 2 m temperature, 2024-07-15 12 UTC (N320, 542,080 points)")
ax.set_aspect("equal")
plt.show()

### Data licence and attribution

Copyright © 2011-2024 European Centre for Medium-Range Weather Forecasts (ECMWF). Source: www.ecmwf.int. This data is published under a [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/) licence, following the [ECMWF terms of use](https://apps.ecmwf.int/datasets/licences/general/). ECMWF does not accept any liability whatsoever for any error or omission in the data, their availability, or for any loss or damage arising from their use.

Beyond Weather modified the data: it selected the variables and levels, converted them from GRIB to Icechunk Zarr, renamed the variables, and bit-rounded the values. See [`LICENSE.txt`](https://beyond-weather-ecmwf-ifs-analysis.s3.us-east-2.amazonaws.com/LICENSE.txt) for the full text. If you use or redistribute the data, credit ECMWF with this wording and state any further changes you make.